# Workshop: vlastný model na dofarbovanie čiernobielych fotografií

Cieľ notebooku je natrénovať jednoduchý vlastný neurónový model na úlohu **image colorization**.

Pipeline:

1. načítam farebné obrázky z datasetu,
2. z farebných obrázkov vytvorím čiernobiely vstup,
3. model dostane grayscale obrázok a učí sa predikovať farebný RGB obrázok,
4. výsledok ukážem na 10 obrázkoch,
5. uložím výstupy pre 5-slidovú prezentáciu.

Toto je vlastný trénovaný model, nie hotový stiahnutý predtrénovaný model.


In [ ]:
from pathlib import Path
import random
import time
import math

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms
from tqdm.auto import tqdm

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())


## 1. Nastavenie ciest a hyperparametrov

Ak chceš rýchly test, nechaj menší počet obrázkov a epoch.  
Ak chceš lepší výsledok, zvýš `MAX_IMAGES` a `EPOCHS`.


In [ ]:
# Notebook môže byť spustený buď z root priečinka projektu, alebo z priečinka notebooks.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name.lower() == "notebooks" else Path.cwd()

DATASET_DIR = PROJECT_ROOT / "data" / "dataset"
SAMPLE_DIR = PROJECT_ROOT / "data" / "sample_color"

OUTPUT_DIR = PROJECT_ROOT / "outputs"
COLORIZED_DIR = OUTPUT_DIR / "colorized"
COMPARISON_DIR = OUTPUT_DIR / "comparisons"
CHECKPOINT_DIR = PROJECT_ROOT / "models" / "own_colorization"

for p in [OUTPUT_DIR, COLORIZED_DIR, COMPARISON_DIR, CHECKPOINT_DIR]:
    p.mkdir(parents=True, exist_ok=True)

# Kvalitnejšie nastavenie: pomalšie, ale lepší výstup.
IMAGE_SIZE = 256
MAX_IMAGES = 3000       # kvalitnejší tréning, dlhšie trvá
BATCH_SIZE = 4
EPOCHS = 25             # checkpoint sa uloží po každej epoche
LEARNING_RATE = 1e-3
SEED = 42
NUM_WORKERS = 0         # na Windows nechaj 0

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATASET_DIR:", DATASET_DIR)
print("SAMPLE_DIR:", SAMPLE_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("Device:", DEVICE)


## 2. Načítanie datasetu

Použijem farebné obrázky, pretože z nich viem automaticky vytvoriť dvojice:

- vstup modelu: čiernobiely obrázok,
- target: pôvodný farebný obrázok.

To je supervised learning, pretože pre každý vstup existuje očakávaný výstup.


In [ ]:
def collect_images(folder: Path):
    exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
    if not folder.exists():
        return []
    return sorted([p for p in folder.rglob("*") if p.suffix.lower() in exts])

all_image_paths = collect_images(DATASET_DIR)
if len(all_image_paths) == 0:
    all_image_paths = collect_images(SAMPLE_DIR)

print("Found images:", len(all_image_paths))

if len(all_image_paths) == 0:
    raise FileNotFoundError("Nenašli sa žiadne obrázky. Skontroluj data/dataset alebo data/sample_color.")

random.shuffle(all_image_paths)
image_paths = all_image_paths[:min(MAX_IMAGES, len(all_image_paths))]

print("Used images:", len(image_paths))
print("First files:")
for p in image_paths[:5]:
    print(" -", p.name)


In [ ]:
# Ukážka niekoľkých farebných obrázkov z datasetu
sample_paths = image_paths[:6]

plt.figure(figsize=(12, 7))
for i, path in enumerate(sample_paths):
    img = Image.open(path).convert("RGB")
    plt.subplot(2, 3, i + 1)
    plt.imshow(img)
    plt.title(path.name)
    plt.axis("off")
plt.tight_layout()
plt.show()


## 3. Dataset trieda

Model bude mať:

- **input**: grayscale tensor s tvarom `[1, H, W]`,
- **target**: RGB tensor s tvarom `[3, H, W]`.

Obrázky sú škálované do rozsahu `[0, 1]`.


In [ ]:
class ColorizationDataset(Dataset):
    def __init__(self, image_paths, image_size=96):
        self.image_paths = list(image_paths)

        self.to_rgb_tensor = transforms.Compose([
            transforms.Resize((image_size, image_size)),
            transforms.ToTensor()
        ])

        self.to_gray_tensor = transforms.Compose([
            transforms.Resize((image_size, image_size)),
            transforms.Grayscale(num_output_channels=1),
            transforms.ToTensor()
        ])

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        path = self.image_paths[idx]
        img = Image.open(path).convert("RGB")

        gray = self.to_gray_tensor(img)
        rgb = self.to_rgb_tensor(img)

        return gray, rgb, path.name


dataset = ColorizationDataset(image_paths, image_size=IMAGE_SIZE)

train_size = int(0.85 * len(dataset))
test_size = len(dataset) - train_size

generator = torch.Generator().manual_seed(SEED)
train_ds, test_ds = random_split(dataset, [train_size, test_size], generator=generator)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

print("Train images:", len(train_ds))
print("Test images:", len(test_ds))


In [ ]:
# Kontrola jedného batchu
gray_batch, rgb_batch, names = next(iter(train_loader))

print("Gray batch:", gray_batch.shape)
print("RGB batch:", rgb_batch.shape)

plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.imshow(gray_batch[0].permute(1, 2, 0).squeeze(), cmap="gray")
plt.title("Model input: grayscale")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(rgb_batch[0].permute(1, 2, 0))
plt.title("Target: original RGB")
plt.axis("off")

plt.tight_layout()
plt.show()


## 4. Model: jednoduchý CNN encoder-decoder so skip connection

Použijem malý vlastný model podobný zjednodušenému U-Netu.

Prečo tento typ modelu:

- CNN vie pracovať s priestorovou štruktúrou obrazu,
- encoder zmenšuje rozmer a učí sa reprezentáciu,
- decoder rekonštruuje farebný obrázok,
- skip connection pomáha zachovať hrany a lokálne detaily.


In [ ]:
class ConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)


class SmallUNetColorizer(nn.Module):
    def __init__(self):
        super().__init__()

        # Encoder
        self.enc1 = ConvBlock(1, 32)
        self.pool1 = nn.MaxPool2d(2)

        self.enc2 = ConvBlock(32, 64)
        self.pool2 = nn.MaxPool2d(2)

        self.bottleneck = ConvBlock(64, 128)

        # Decoder
        self.up2 = nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False)
        self.dec2 = ConvBlock(128 + 64, 64)

        self.up1 = nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False)
        self.dec1 = ConvBlock(64 + 32, 32)

        self.out = nn.Sequential(
            nn.Conv2d(32, 3, kernel_size=1),
            nn.Sigmoid()
        )

    def forward(self, x):
        e1 = self.enc1(x)
        x = self.pool1(e1)

        e2 = self.enc2(x)
        x = self.pool2(e2)

        x = self.bottleneck(x)

        x = self.up2(x)
        x = torch.cat([x, e2], dim=1)
        x = self.dec2(x)

        x = self.up1(x)
        x = torch.cat([x, e1], dim=1)
        x = self.dec1(x)

        return self.out(x)


model = SmallUNetColorizer().to(DEVICE)

num_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(model)
print("Parameters:", num_params)
print("Trainable parameters:", trainable_params)


## 5. Tréning

Použijem `L1Loss`, pretože pri rekonštrukcii obrazu často dáva menej rozmazané výsledky ako MSE.  
Model sa učí minimalizovať rozdiel medzi predikovaným RGB obrázkom a pôvodným farebným obrázkom.

**Dôležité:** checkpoint sa uloží automaticky po každej dokončenej epoche. Ak sa notebook vypne alebo tréning prerušíš, ďalšie spustenie vie pokračovať od poslednej uloženej epochy.


In [ ]:
criterion = nn.L1Loss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

RESUME_CHECKPOINT_PATH = CHECKPOINT_DIR / "resume_checkpoint.pth"
FINAL_CHECKPOINT_PATH = CHECKPOINT_DIR / "small_unet_colorizer.pth"

# Ak chceš začať úplne od nuly, nastav na False alebo vymaž resume_checkpoint.pth.
RESUME_TRAINING = True

history = {
    "train_loss": [],
    "test_loss": []
}

def evaluate(model, loader):
    model.eval()
    total_loss = 0.0
    total_items = 0

    with torch.no_grad():
        for gray, rgb, _ in loader:
            gray = gray.to(DEVICE)
            rgb = rgb.to(DEVICE)

            pred = model(gray)
            loss = criterion(pred, rgb)

            batch_size = gray.size(0)
            total_loss += loss.item() * batch_size
            total_items += batch_size

    return total_loss / max(total_items, 1)


def save_training_checkpoint(epoch_number):
    """Uloží kompletný stav po dokončenej epoche."""
    checkpoint = {
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "image_size": IMAGE_SIZE,
        "max_images": MAX_IMAGES,
        "batch_size": BATCH_SIZE,
        "epochs_total": EPOCHS,
        "epochs": EPOCHS,
        "completed_epochs": epoch_number,
        "epochs_finished": epoch_number,
        "learning_rate": LEARNING_RATE,
        "history": history,
    }

    # checkpoint na pokračovanie tréningu
    torch.save(checkpoint, RESUME_CHECKPOINT_PATH)

    # checkpoint používaný pri finálnom testovaní / inference
    torch.save(checkpoint, FINAL_CHECKPOINT_PATH)

    print(f"Auto-saved checkpoint after epoch {epoch_number}:")
    print(" -", RESUME_CHECKPOINT_PATH)
    print(" -", FINAL_CHECKPOINT_PATH)


# ---------- Automatické pokračovanie ----------
START_EPOCH = 1
completed_epochs = 0

if RESUME_TRAINING and RESUME_CHECKPOINT_PATH.exists():
    checkpoint = torch.load(RESUME_CHECKPOINT_PATH, map_location=DEVICE)

    saved_image_size = checkpoint.get("image_size")
    saved_max_images = checkpoint.get("max_images")
    saved_batch_size = checkpoint.get("batch_size")

    config_matches = (
        saved_image_size == IMAGE_SIZE and
        saved_max_images == MAX_IMAGES and
        saved_batch_size == BATCH_SIZE
    )

    if config_matches:
        model.load_state_dict(checkpoint["model_state_dict"])

        if "optimizer_state_dict" in checkpoint:
            optimizer.load_state_dict(checkpoint["optimizer_state_dict"])

        history = checkpoint.get("history", history)

        # Dôležité: nepoužívame len len(history), ale uložené číslo dokončenej epochy.
        completed_epochs = int(
            checkpoint.get(
                "completed_epochs",
                checkpoint.get("epochs_finished", len(history.get("train_loss", [])))
            )
        )

        START_EPOCH = completed_epochs + 1

        print("Loaded resume checkpoint:", RESUME_CHECKPOINT_PATH)
        print("Completed epochs:", completed_epochs)
        print("Next epoch:", START_EPOCH)
    else:
        print("Existing resume checkpoint was found, but it uses different training settings.")
        print("Saved config:", {
            "image_size": saved_image_size,
            "max_images": saved_max_images,
            "batch_size": saved_batch_size
        })
        print("Current config:", {
            "image_size": IMAGE_SIZE,
            "max_images": MAX_IMAGES,
            "batch_size": BATCH_SIZE
        })
        print("Starting new high-quality training from epoch 1.")
else:
    print("Starting training from epoch 1.")


# ---------- Tréning ----------
if START_EPOCH > EPOCHS:
    print(f"Training already finished: {completed_epochs}/{EPOCHS} epochs.")
    print("If you want to train more, increase EPOCHS or delete resume_checkpoint.pth.")
else:
    start_time = time.time()

    try:
        for epoch in range(START_EPOCH, EPOCHS + 1):
            model.train()
            running_loss = 0.0
            running_items = 0

            progress = tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS}", leave=False)

            for gray, rgb, _ in progress:
                gray = gray.to(DEVICE)
                rgb = rgb.to(DEVICE)

                optimizer.zero_grad()
                pred = model(gray)
                loss = criterion(pred, rgb)
                loss.backward()
                optimizer.step()

                batch_size = gray.size(0)
                running_loss += loss.item() * batch_size
                running_items += batch_size

                progress.set_postfix(loss=loss.item())

            train_loss = running_loss / max(running_items, 1)
            test_loss = evaluate(model, test_loader)

            history["train_loss"].append(train_loss)
            history["test_loss"].append(test_loss)

            completed_epochs = epoch

            print(f"Epoch {epoch:02d}/{EPOCHS} | train L1: {train_loss:.4f} | test L1: {test_loss:.4f}")

            # Automatické uloženie po každej dokončenej epoche.
            save_training_checkpoint(epoch)

    except KeyboardInterrupt:
        print("")
        print("Training interrupted manually.")
        print(f"Last fully completed and saved epoch: {completed_epochs}")
        print("You can safely resume later by running this notebook again.")

    elapsed = time.time() - start_time
    print(f"Training block finished in {elapsed/60:.1f} minutes.")


In [ ]:
# Kontrola uložených checkpointov
print("Resume checkpoint:", RESUME_CHECKPOINT_PATH, RESUME_CHECKPOINT_PATH.exists())
print("Final/inference checkpoint:", FINAL_CHECKPOINT_PATH, FINAL_CHECKPOINT_PATH.exists())

if FINAL_CHECKPOINT_PATH.exists():
    saved = torch.load(FINAL_CHECKPOINT_PATH, map_location="cpu")
    print("Completed epochs:", saved.get("completed_epochs", saved.get("epochs_finished")))
    print("Total planned epochs:", saved.get("epochs_total", saved.get("epochs")))
    print("Image size:", saved.get("image_size"))
    print("History train length:", len(saved.get("history", {}).get("train_loss", [])))
    print("History test length:", len(saved.get("history", {}).get("test_loss", [])))


In [ ]:
# Graf loss počas tréningu
plt.figure(figsize=(8, 5))
plt.plot(history["train_loss"], label="train L1 loss")
plt.plot(history["test_loss"], label="test L1 loss")
plt.xlabel("Epoch")
plt.ylabel("L1 loss")
plt.title("Training history")
plt.legend()
plt.grid(True)

loss_plot_path = OUTPUT_DIR / "training_loss.png"
plt.savefig(loss_plot_path, dpi=150, bbox_inches="tight")
plt.show()

print("Saved loss plot to:", loss_plot_path)


## 6. Výsledky na 10 obrázkoch

Prezentácia má ukázať výsledok na 10 obrázkoch.  
Každý výstup obsahuje:

1. pôvodný farebný obrázok,
2. čiernobiely vstup,
3. dofarbený výstup modelu.


In [ ]:
def tensor_to_image(tensor):
    # tensor: [C, H, W], values 0..1
    tensor = tensor.detach().cpu().clamp(0, 1)
    arr = tensor.permute(1, 2, 0).numpy()
    arr = (arr * 255).astype(np.uint8)
    return Image.fromarray(arr)

def gray_tensor_to_image(tensor):
    tensor = tensor.detach().cpu().clamp(0, 1)
    arr = tensor.squeeze(0).numpy()
    arr = (arr * 255).astype(np.uint8)
    return Image.fromarray(arr, mode="L")

def save_comparison(gray, pred, target, name, index):
    gray_img = gray_tensor_to_image(gray)
    pred_img = tensor_to_image(pred)
    target_img = tensor_to_image(target)

    pred_path = COLORIZED_DIR / f"{index:02d}_{Path(name).stem}_colorized.png"
    cmp_path = COMPARISON_DIR / f"{index:02d}_{Path(name).stem}_comparison.png"

    pred_img.save(pred_path)

    fig = plt.figure(figsize=(10, 4))

    plt.subplot(1, 3, 1)
    plt.imshow(target_img)
    plt.title("Original")
    plt.axis("off")

    plt.subplot(1, 3, 2)
    plt.imshow(gray_img, cmap="gray")
    plt.title("Grayscale input")
    plt.axis("off")

    plt.subplot(1, 3, 3)
    plt.imshow(pred_img)
    plt.title("Model output")
    plt.axis("off")

    plt.tight_layout()
    plt.savefig(cmp_path, dpi=150, bbox_inches="tight")
    plt.show()
    plt.close(fig)

    return pred_path, cmp_path


model.eval()

saved = []
count = 0

with torch.no_grad():
    for gray_batch, rgb_batch, names in test_loader:
        gray_batch = gray_batch.to(DEVICE)
        rgb_batch = rgb_batch.to(DEVICE)

        pred_batch = model(gray_batch)

        for i in range(gray_batch.size(0)):
            if count >= 10:
                break

            pred_path, cmp_path = save_comparison(
                gray_batch[i],
                pred_batch[i],
                rgb_batch[i],
                names[i],
                count + 1
            )

            saved.append((pred_path, cmp_path))
            count += 1

        if count >= 10:
            break

print(f"Saved {len(saved)} result comparisons.")
print("Colorized outputs:", COLORIZED_DIR)
print("Comparisons:", COMPARISON_DIR)


In [ ]:
# Kontrola počtu uložených výstupov
colorized_count = len(list(COLORIZED_DIR.glob("*.png")))
comparison_count = len(list(COMPARISON_DIR.glob("*.png")))

print("Colorized image count:", colorized_count)
print("Comparison image count:", comparison_count)


## 7. Krátke zhodnotenie

Doplň podľa reálnych výsledkov po spustení notebooku.


In [ ]:
# Robustné zhrnutie experimentu
# Funguje aj po reštarte kernelu, ak existuje uložený checkpoint.

from pathlib import Path
import torch

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name.lower() == "notebooks" else Path.cwd()
OUTPUT_DIR = PROJECT_ROOT / "outputs"
CHECKPOINT_DIR = PROJECT_ROOT / "models" / "own_colorization"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

checkpoint_path = CHECKPOINT_DIR / "small_unet_colorizer.pth"

if checkpoint_path.exists():
    checkpoint = torch.load(checkpoint_path, map_location="cpu")
    saved_history = checkpoint.get("history", {})
    saved_epochs = checkpoint.get("completed_epochs", checkpoint.get("epochs_finished", checkpoint.get("epochs", "unknown")))
    saved_max_images = checkpoint.get("max_images", "unknown")
    saved_image_size = checkpoint.get("image_size", "unknown")
else:
    checkpoint = {}
    saved_history = history if "history" in globals() else {}
    saved_epochs = EPOCHS if "EPOCHS" in globals() else "unknown"
    saved_max_images = len(image_paths) if "image_paths" in globals() else "unknown"
    saved_image_size = IMAGE_SIZE if "IMAGE_SIZE" in globals() else "unknown"

if "image_paths" in globals():
    used_images = len(image_paths)
elif isinstance(saved_max_images, int):
    used_images = saved_max_images
else:
    used_images = "unknown"

if "train_ds" in globals() and "test_ds" in globals():
    train_count = len(train_ds)
    test_count = len(test_ds)
elif isinstance(used_images, int):
    train_count = int(0.85 * used_images)
    test_count = used_images - train_count
else:
    train_count = "unknown"
    test_count = "unknown"

train_losses = saved_history.get("train_loss", [])
test_losses = saved_history.get("test_loss", [])

final_train_loss = train_losses[-1] if len(train_losses) > 0 else None
final_test_loss = test_losses[-1] if len(test_losses) > 0 else None

final_train_loss_txt = f"{final_train_loss:.4f}" if final_train_loss is not None else "unknown"
final_test_loss_txt = f"{final_test_loss:.4f}" if final_test_loss is not None else "unknown"

summary = f"""
Zhrnutie experimentu:

- Použitý dataset: {used_images} farebných obrázkov.
- Trénovací set: {train_count} obrázkov.
- Testovací set: {test_count} obrázkov.
- Veľkosť obrázkov: {saved_image_size} x {saved_image_size}.
- Vstup modelu: grayscale obrázok s 1 kanálom.
- Výstup modelu: RGB obrázok s 3 kanálmi.
- Architektúra: jednoduchý CNN encoder-decoder so skip connections.
- Loss funkcia: L1 loss.
- Počet epoch: {saved_epochs}.
- Finálny train loss: {final_train_loss_txt}.
- Finálny test loss: {final_test_loss_txt}.
- Model checkpoint: models/own_colorization/small_unet_colorizer.pth
- Interné výsledky: outputs/colorized a outputs/comparisons.
- Externé testovanie: input_images -> outputs/teacher_colorized a outputs/teacher_comparisons.

Interpretácia:
Model sa naučil základnú transformáciu z čiernobieleho vstupu na farebný výstup.
Výsledky môžu byť menej sýte alebo nie vždy farebne presné, pretože dofarbovanie je nejednoznačná úloha:
z jednej grayscale fotografie sa nedá jednoznačne určiť pôvodná farba objektov.
"""

print(summary)

summary_path = OUTPUT_DIR / "experiment_summary.txt"
summary_path.write_text(summary, encoding="utf-8")
print("Saved summary to:", summary_path)


In [ ]:
## 8. Test na externých čiernobielych obrázkoch

# Táto bunka je samostatná: funguje aj po reštarte kernelu.
# Simuluje hodnotenie učiteľom:
# 1. vlož 10 čiernobielych obrázkov do input_images/
# 2. spusti túto bunku
# 3. výsledky nájdeš v outputs/teacher_colorized/ a outputs/teacher_comparisons/
#
# Dôležité:
# Model predikuje farby v tréningovom rozlíšení, ale finálny výstup sa uloží v pôvodnom rozlíšení vstupu.
# Ostrý detail sa berie z pôvodného grayscale obrázka a farba z modelu.

from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
from torchvision import transforms

# ---------- CESTY ----------
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name.lower() == "notebooks" else Path.cwd()

OUTPUT_DIR = PROJECT_ROOT / "outputs"
CHECKPOINT_DIR = PROJECT_ROOT / "models" / "own_colorization"

TEACHER_INPUT_DIR = PROJECT_ROOT / "input_images"
TEACHER_COLORIZED_DIR = OUTPUT_DIR / "teacher_colorized"
TEACHER_COMPARISON_DIR = OUTPUT_DIR / "teacher_comparisons"

TEACHER_COLORIZED_DIR.mkdir(parents=True, exist_ok=True)
TEACHER_COMPARISON_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DEVICE:", DEVICE)

# ---------- POMOCNÉ FUNKCIE ----------
def collect_images_external(folder: Path):
    exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
    if not folder.exists():
        return []
    return sorted([p for p in folder.rglob("*") if p.suffix.lower() in exts])

def tensor_to_image_external(tensor):
    tensor = tensor.detach().cpu().clamp(0, 1)
    arr = tensor.permute(1, 2, 0).numpy()
    arr = (arr * 255).astype(np.uint8)
    return Image.fromarray(arr)

# ---------- MODEL ----------
class ConvBlockExternal(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)


class SmallUNetColorizerExternal(nn.Module):
    def __init__(self):
        super().__init__()

        self.enc1 = ConvBlockExternal(1, 32)
        self.pool1 = nn.MaxPool2d(2)

        self.enc2 = ConvBlockExternal(32, 64)
        self.pool2 = nn.MaxPool2d(2)

        self.bottleneck = ConvBlockExternal(64, 128)

        self.up2 = nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False)
        self.dec2 = ConvBlockExternal(128 + 64, 64)

        self.up1 = nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False)
        self.dec1 = ConvBlockExternal(64 + 32, 32)

        self.out = nn.Sequential(
            nn.Conv2d(32, 3, kernel_size=1),
            nn.Sigmoid()
        )

    def forward(self, x):
        e1 = self.enc1(x)
        x = self.pool1(e1)

        e2 = self.enc2(x)
        x = self.pool2(e2)

        x = self.bottleneck(x)

        x = self.up2(x)
        x = torch.cat([x, e2], dim=1)
        x = self.dec2(x)

        x = self.up1(x)
        x = torch.cat([x, e1], dim=1)
        x = self.dec1(x)

        return self.out(x)

# ---------- NAČÍTANIE MODELU ----------
checkpoint_path = CHECKPOINT_DIR / "small_unet_colorizer.pth"

if not checkpoint_path.exists():
    raise FileNotFoundError(f"Model checkpoint neexistuje: {checkpoint_path}. Najprv spusti tréningovú časť notebooku.")

checkpoint = torch.load(checkpoint_path, map_location=DEVICE)
IMAGE_SIZE = checkpoint.get("image_size", 256)

model_external = SmallUNetColorizerExternal().to(DEVICE)
model_external.load_state_dict(checkpoint["model_state_dict"])
model_external.eval()

print("Loaded model:", checkpoint_path)
print("Model training IMAGE_SIZE:", IMAGE_SIZE)

# ---------- EXTERNÉ OBRÁZKY ----------
external_paths = collect_images_external(TEACHER_INPUT_DIR)

print("Found external images:", len(external_paths))
for p in external_paths[:10]:
    print(" -", p.name)

if len(external_paths) == 0:
    raise FileNotFoundError("input_images/ je prázdny. Najprv tam vlož 10 čiernobielych obrázkov.")

external_paths = external_paths[:10]

external_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.Grayscale(num_output_channels=1),
    transforms.ToTensor()
])

def save_teacher_result_highres(original_path, pred_tensor, original_name, index):
    # Pôvodný ostrý čiernobiely vstup v pôvodnom rozlíšení
    original_gray = Image.open(original_path).convert("L")
    original_size = original_gray.size

    # Predikovaná farba z modelu v nižšom rozlíšení
    pred_img_low = tensor_to_image_external(pred_tensor)

    # Farbu zväčšíme na pôvodnú veľkosť
    pred_img_high = pred_img_low.resize(original_size, Image.Resampling.BICUBIC)

    # Ostrý detail z grayscale obrázka
    gray_rgb = Image.merge("RGB", (original_gray, original_gray, original_gray))

    # Kombinácia: detail z grayscale + farebný nádych z modelu
    final_img = Image.blend(gray_rgb, pred_img_high, alpha=0.65)

    colorized_path = TEACHER_COLORIZED_DIR / f"{index:02d}_{Path(original_name).stem}_colorized_highres.png"
    comparison_path = TEACHER_COMPARISON_DIR / f"{index:02d}_{Path(original_name).stem}_teacher_comparison_highres.png"

    final_img.save(colorized_path)

    fig = plt.figure(figsize=(12, 5))

    plt.subplot(1, 2, 1)
    plt.imshow(original_gray, cmap="gray")
    plt.title("External grayscale input")
    plt.axis("off")

    plt.subplot(1, 2, 2)
    plt.imshow(final_img)
    plt.title("High-res colorized output")
    plt.axis("off")

    plt.tight_layout()
    plt.savefig(comparison_path, dpi=150, bbox_inches="tight")
    plt.show()
    plt.close(fig)

    return colorized_path, comparison_path

# ---------- INFERENCE ----------
saved_teacher_results = []

with torch.no_grad():
    for index, path in enumerate(external_paths, start=1):
        img = Image.open(path).convert("RGB")
        gray = external_transform(img).unsqueeze(0).to(DEVICE)

        pred = model_external(gray)

        colorized_path, comparison_path = save_teacher_result_highres(
            path,
            pred[0],
            path.name,
            index
        )

        saved_teacher_results.append((colorized_path, comparison_path))

print("Saved teacher-style high-res results:", len(saved_teacher_results))
print("Colorized outputs:", TEACHER_COLORIZED_DIR)
print("Comparison outputs:", TEACHER_COMPARISON_DIR)


## 9. Body do 5-slidovej prezentácie

**Slide 1 – Zadanie a cieľ**  
Dofarbiť čiernobiele fotografie pomocou vlastného natrénovaného modelu.

**Slide 2 – Dataset a príprava dát**  
Použil som farebné obrázky, z ktorých som vytvoril grayscale vstupy. Pôvodné RGB obrázky slúžili ako target.

**Slide 3 – Model**  
Jednoduchý CNN encoder-decoder / mini U-Net: grayscale vstup → latentná reprezentácia → RGB výstup.

**Slide 4 – Výsledky**  
Ukážka 10 obrázkov: original / grayscale / model output.

**Slide 5 – Záver a limity**  
Model funguje ako základná colorization pipeline, ale farby nemusia byť fakticky správne. Úloha je nejednoznačná a lepší výsledok by vyžadoval väčší dataset, dlhší tréning alebo silnejšiu architektúru.
